# Setting up the dependencies
Here, we’re setting up a minimal environment to interact with the OpenAI API. We securely load the API key at runtime using getpass, initialize the client, and define a lightweight chat wrapper to send system and user prompts to the model (gpt-4o-mini). This keeps our experimentation loop clean and reusable while focusing only on prompt variations.

The helper functions (section and divider) are just for formatting outputs, making it easier to compare baseline vs. improved prompts side by side. If you don’t already have an API key, you can create one from the official dashboard here: https://platform.openai.com/api-keys


In [2]:
import json
from openai import OpenAI
import os
from getpass import getpass

In [3]:
os.environ['OPENAI_API_KEY'] = getpass('Enter OpenAI API Key: ')

Enter OpenAI API Key: ··········


In [4]:
client = OpenAI()
MODEL = "gpt-4o-mini"


def chat(system: str, user: str, **kwargs) -> str:
    """Minimal wrapper around the chat completions endpoint."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": user},
        ],
        **kwargs,
    )
    return response.choices[0].message.content


def section(title: str) -> None:
    print()
    print("=" * 60)
    print(f"  {title}")
    print("=" * 60)


def divider(label: str) -> None:
    print(f"\n── {label} {'─' * (54 - len(label))}")

# Role-Specific Prompting
Language models are trained on a wide mix of domains—security, marketing, legal, engineering, and more. When you don’t specify a role, the model pulls from all of them, which leads to answers that are generally correct but somewhat generic. Role-specific prompting fixes this by assigning a persona in the system prompt (e.g., “You are a senior application security researcher”). This acts like a filter, pushing the model to respond using the language, priorities, and reasoning style of that domain.

In this example, both responses identify the XSS risk and recommend HttpOnly cookies — the underlying facts are identical. The difference is in how the model frames the problem. The baseline treats localStorage as a configuration choice with tradeoffs. The role-specific response treats it as an attack surface: it reasons about what an attacker can do once XSS is present, not just that XSS is theoretically possible. That shift in framing — from "here are the risks" to "here is what an attacker does with those risks" — is the conditioning effect in action. No new information was provided. The prompt just changed which part of the model's knowledge got weighted.


In [5]:
section("TECHNIQUE 1 — Role-Specific Prompting")

QUESTION = "Our web app stores session tokens in localStorage. Is this a problem?"

baseline_1 = chat(
    system="You are a helpful assistant.",
    user=QUESTION,
)

role_specific = chat(
    system=(
        "You are a senior application security researcher specializing in "
        "web authentication vulnerabilities. You think in terms of attack "
        "surface, threat models, and OWASP guidelines."
    ),
    user=QUESTION,
)

divider("Baseline")
print(baseline_1)

divider("Role-specific (security researcher)")
print(role_specific)


  TECHNIQUE 1 — Role-Specific Prompting

── Baseline ──────────────────────────────────────────────
Storing session tokens in `localStorage` can present security risks. Here are some considerations to help you evaluate whether this is a problem for your web app:

### Risks of Storing Tokens in `localStorage`:

1. **XSS Vulnerability**: 
   - If your application is vulnerable to Cross-Site Scripting (XSS) attacks, an attacker could potentially inject malicious scripts. These scripts can access `localStorage` and steal tokens stored there, allowing them to impersonate the user.

2. **No Automatic Expiration**:
   - `localStorage` does not automatically expire data. If a token is stored without an expiration policy, it may remain until explicitly cleared, increasing the risk if the token is compromised.

3. **Limited Accessibility**:
   - `localStorage` can only be accessed by the same origin (domain, protocol, and port). However, if your application has multiple subdomains, managing tok

# Negative Prompting
Negative prompting focuses on telling the model what not to do. By default, LLMs follow patterns learned during training and RLHF—they add friendly openings, analogies, hedging (“it depends”), and closing summaries. While this makes responses feel helpful, it often adds unnecessary noise in technical contexts. Negative prompting works by removing these defaults. Instead of just describing the desired output, you also restrict unwanted behaviors, which narrows the model’s output space and leads to more precise responses.

In the output, the difference is immediately visible. The baseline response stretches into a longer, structured explanation with analogies, headers, and a redundant conclusion. The negatively prompted version delivers the same core information in a much shorter form—direct, concise, and without filler. Nothing essential is lost; the prompt simply removes the model’s tendency to over-explain and pad the response.


In [6]:
section("TECHNIQUE 2 — Negative Prompting")

TOPIC = "Explain what a database index is and when you'd use one."

baseline_2 = chat(
    system="You are a helpful assistant.",
    user=TOPIC,
)

negative = chat(
    system=(
        "You are a senior backend engineer writing internal documentation.\n"
        "Rules:\n"
        "- Do NOT use marketing language or filler phrases like 'great question' or 'certainly'.\n"
        "- Do NOT include caveats like 'it depends' without immediately resolving them.\n"
        "- Do NOT use analogies unless they are necessary. If you use one, keep it to one sentence.\n"
        "- Do NOT pad the response — if you've made the point, stop.\n"
    ),
    user=TOPIC,
)

divider("Baseline")
print(baseline_2)

divider("With negative prompting")
print(negative)


  TECHNIQUE 2 — Negative Prompting

── Baseline ──────────────────────────────────────────────
A database index is a data structure that improves the speed of data retrieval operations on a database table. It works similarly to an index in a book, which allows you to find specific information quickly without having to read through every page. 

### How Database Indexes Work
1. **Structure**: Indexes are typically implemented as B-trees or hash tables, allowing rapid lookup of data. Each index is associated with one or more columns in a database table.
   
2. **Organization**: When a database is indexed, the database management system (DBMS) creates a separate structure containing the values of the indexed column(s) and pointers to the corresponding records in the table.

3. **Search Performance**: When a query is executed, instead of scanning the entire table, the database can use the index to quickly locate the rows that match the search criteria.

### When to Use a Database Index
1.

# JSON Prompting (Schema-Constrained Output)
JSON prompting becomes important when LLM outputs need to be consumed by code rather than just read by humans. Free-form responses are inconsistent—structure varies, key details are embedded in paragraphs, and small wording changes break parsing logic. By defining a JSON schema in the prompt, you turn structure into a hard constraint. This not only standardizes the output format but also forces the model to organize its reasoning into clearly defined fields like pros, cons, sentiment, and rating.
In the output, the difference is clear. The baseline response is readable but unstructured—pros, cons, and sentiment are mixed into narrative text, making it difficult to parse. The JSON-prompted version, however, returns clean, well-defined fields that can be directly loaded and used in code without any post-processing. Information that was previously implied is now explicit and separated, making the output easy to store, query, and compare at scale.


In [7]:
section("TECHNIQUE 3 — JSON Prompting")

REVIEW = """
Honestly mixed feelings about this laptop. The display is stunning — easily the best I've
seen at this price range — and the keyboard is surprisingly comfortable for long sessions.
Battery life, on the other hand, barely gets me through a 6-hour workday, which is
disappointing. Fan noise under load is also pretty aggressive. For light work it's great,
but I wouldn't recommend it for anyone who needs to run heavy software.
"""

SCHEMA = """
{
  "overall_sentiment": "positive | negative | mixed",
  "rating": <integer 1-5>,
  "pros": ["<string>", ...],
  "cons": ["<string>", ...],
  "recommended_for": "<string describing ideal user>",
  "not_recommended_for": "<string describing user who should avoid>"
}
"""

baseline_3 = chat(
    system="You are a helpful assistant.",
    user=f"Summarize this product review:\n\n{REVIEW}",
)

json_output = chat(
    system=(
        "You are a product review parser. Extract structured information from reviews.\n"
        "You MUST return only a valid JSON object. No preamble, no explanation, no markdown fences.\n"
        f"The JSON must match this schema exactly:\n{SCHEMA}"
    ),
    user=f"Parse this review:\n\n{REVIEW}",
)

divider("Baseline (free-form)")
print(baseline_3)

divider("JSON prompting (raw output)")
print(json_output)

divider("Parsed & usable in code")
parsed = json.loads(json_output)
print(f"Sentiment         : {parsed['overall_sentiment']}")
print(f"Rating            : {parsed['rating']}/5")
print(f"Pros              : {', '.join(parsed['pros'])}")
print(f"Cons              : {', '.join(parsed['cons'])}")
print(f"Recommended for   : {parsed['recommended_for']}")
print(f"Avoid if          : {parsed['not_recommended_for']}")


  TECHNIQUE 3 — JSON Prompting

── Baseline (free-form) ──────────────────────────────────
The reviewer has mixed feelings about the laptop. They praise the stunning display and comfortable keyboard, noting that it excels for light work. However, they express disappointment with the battery life, which struggles to last a full 6-hour workday, and mention that the fan noise is quite loud under load. Overall, they do not recommend it for users needing to run heavy software.

── JSON prompting (raw output) ───────────────────────────
{
  "overall_sentiment": "mixed",
  "rating": 3,
  "pros": ["stunning display", "comfortable keyboard for long sessions"],
  "cons": ["poor battery life", "aggressive fan noise under load"],
  "recommended_for": "users who do light work",
  "not_recommended_for": "users who need to run heavy software"
}

── Parsed & usable in code ───────────────────────────────
Sentiment         : mixed
Rating            : 3/5
Pros              : stunning display, comfortab

# Attentive Reasoning Queries (ARQ)
Attentive Reasoning Queries (ARQ) build on chain-of-thought prompting but remove its biggest weakness—unstructured reasoning. In standard CoT, the model decides what to focus on, which can lead to gaps or irrelevant details. ARQ replaces this with a fixed set of domain-specific questions that the model must answer in order. This ensures that all critical aspects are covered, shifting control from the model to the prompt designer. Instead of just guiding how the model thinks, ARQ defines what it must think about.

In the output, the difference shows up as discipline and coverage. The baseline CoT response identifies key issues but drifts into less relevant areas and misses deeper analysis in places. The ARQ version, however, systematically addresses each required point—clearly isolating vulnerabilities, handling edge cases, and evaluating performance implications. Each question acts as a checkpoint, making the response more structured, complete, and easier to audit.


In [8]:
section("TECHNIQUE 4 — Attentive Reasoning Queries (ARQ)")

CODE_TO_REVIEW = """
def get_user(user_id):
    query = f"SELECT * FROM users WHERE id = {user_id}"
    result = db.execute(query)
    return result[0] if result else None
"""

ARQ_QUESTIONS = """
Before giving your final review, answer each of the following questions in order:

Q1 [Security]: Does this code have any injection vulnerabilities?
               If yes, describe the exact attack vector.
Q2 [Error handling]: What happens if db.execute() throws an exception?
                     Is that acceptable?
Q3 [Performance]: Does this query retrieve more data than necessary?
                  What is the cost at scale?
Q4 [Correctness]: Are there edge cases in the return logic that could
                  cause a silent bug downstream?
Q5 [Fix]: Write a corrected version of the function that addresses
          all issues found above.
"""

baseline_cot = chat(
    system="You are a senior software engineer. Think step by step.",
    user=f"Review this Python function:\n\n{CODE_TO_REVIEW}",
)

arq_result = chat(
    system="You are a senior software engineer conducting a security-aware code review.",
    user=f"Review this Python function:\n\n{CODE_TO_REVIEW}\n\n{ARQ_QUESTIONS}",
)

divider("Baseline (free CoT)")
print(baseline_cot)

divider("ARQ (structured reasoning checklist)")
print(arq_result)


  TECHNIQUE 4 — Attentive Reasoning Queries (ARQ)

── Baseline (free CoT) ───────────────────────────────────
The provided Python function `get_user(user_id)` is intended to retrieve a user record from a database based on the user's ID. Let's review this function step by step, highlighting strengths and weaknesses, as well as suggestions for improvement.

### Code Review

1. **SQL Injection Vulnerability**:
   - The function constructs an SQL query using string interpolation (f-strings). This is dangerous because it opens the possibility for SQL injection attacks if `user_id` can be manipulated by users (e.g., through user input).
   - **Suggestion**: Use parameterized queries to safely include variables in SQL statements. This prevents SQL injection by ensuring input values are treated as data, not as part of the SQL code.

   ```python
   query = "SELECT * FROM users WHERE id = %s"
   result = db.execute(query, (user_id,))
   ```

2. **Error Handling**:
   - The function does not ha

# Verbalized Sampling

Verbalized sampling addresses a key limitation of LLMs: they tend to return a single, confident answer even when multiple interpretations are possible. This happens because alignment training favors decisive outputs. As a result, the model hides its internal uncertainty. Verbalized sampling fixes this by explicitly asking for multiple hypotheses, along with confidence rankings and supporting evidence. Instead of forcing one answer, it surfaces a range of plausible outcomes—all within the prompt, without needing model changes.

In the output, this shifts the result from a single label to a structured diagnostic view. The baseline provides one classification with no indication of uncertainty. The verbalized version, however, lists multiple ranked hypotheses, each with an explanation and a way to validate or reject it. This makes the output more actionable, turning it into a decision-making aid rather than just an answer. The confidence scores themselves aren’t precise probabilities, but they effectively indicate relative likelihood, which is often sufficient for prioritization and downstream workflows.


In [9]:
section("TECHNIQUE 5 — Verbalized Sampling")

SUPPORT_TICKET = """
Hi, I set up my account last week but I can't log in anymore. I tried resetting
my password but the email never arrives. I also tried a different browser. Nothing works.
"""

baseline_5 = chat(
    system="You are a support ticket classifier. Classify the issue.",
    user=f"Ticket:\n{SUPPORT_TICKET}",
)

verbalized = chat(
    system=(
        "You are a support ticket classifier.\n"
        "For each ticket, generate 3 distinct hypotheses about the root cause. "
        "For each hypothesis:\n"
        "  - State the category (Authentication, Email Delivery, Account State, Browser/Client, Other)\n"
        "  - Describe the specific failure mode\n"
        "  - Assign a confidence score from 0.0 to 1.0\n"
        "  - State what additional information would confirm or rule it out\n\n"
        "Order hypotheses by confidence (highest first). "
        "Then provide a recommended first action for the support agent."
    ),
    user=f"Ticket:\n{SUPPORT_TICKET}",
)

divider("Baseline (single answer)")
print(baseline_5)

divider("Verbalized sampling (multiple hypotheses + confidence)")
print(verbalized)


  TECHNIQUE 5 — Verbalized Sampling

── Baseline (single answer) ──────────────────────────────
Classification: Account Access Issue

── Verbalized sampling (multiple hypotheses + confidence) 
### Hypotheses

1. **Hypothesis 1**
   - **Category**: Email Delivery
   - **Specific Failure Mode**: The password reset email is not being delivered due to an issue with the email server or the recipient address being flagged as invalid.
   - **Confidence Score**: 0.85
   - **Additional Information Needed**: Confirmation of the email address used for account setup and checking if any logs show emails being sent or bounced.

2. **Hypothesis 2**
   - **Category**: Authentication
   - **Specific Failure Mode**: There may be an issue with the account credentials or the account being temporarily locked due to multiple failed login attempts.
   - **Confidence Score**: 0.75
   - **Additional Information Needed**: Attempt to retrieve the account status and check if there are any restrictions or securit